In [1]:
from datasets import Dataset, load_dataset

/home/minhnh/python_venv/nlp/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets
from huggingface_hub import create_repo
import pandas as pd
import torch
import transformers
import os
import argparse
import bitsandbytes as bnb
from functools import partial
import os
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, AutoPeftModelForCausalLM, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed, Trainer, TrainingArguments, BitsAndBytesConfig, \
    DataCollatorForLanguageModeling, Trainer, TrainingArguments, BloomForCausalLM, BloomTokenizerFast, pipeline, \
    EarlyStoppingCallback
from accelerate import dispatch_model, infer_auto_device_map
from accelerate.utils import get_balanced_memory
from huggingface_hub.hf_api import HfFolder

2024-10-14 08:10:29.531081: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-14 08:10:30.354699: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [3]:
dataset = load_dataset('ikura31/vo_summarize_section_30k_34k')

In [4]:
dataset = dataset['train']

In [5]:
from g4f.client import Client
from tqdm import tqdm

client = Client()

In [6]:
def prompt(input_):
    response = client.chat.completions.create(
        model="gemini",
        messages=[{"role": "user", "content": f"""dịch đoạn văn bản sau sang tiếng Anh mà vẫn giữ nguyên cấu trúc giống văn bản gốc, giữ nguyên các đề mục và các dấu '+', '-' và '\n', ví dụ mẫu về cách dịch, không sinh thêm kí tự '*' và '**', không sinh thêm các câu nói tương tác của bạn và chỉ sinh thẳng output:
        Input mẫu: 
        'I. Mục đích và Yêu cầu:
        - Cụ thể hóa mục tiêu phát triển bền vững ngành hàng sắn theo Quyết định 1115/QĐBNN-TT của Bộ Nông nghiệp và Phát triển nông thôn.
        - Phù hợp với điều kiện tự nhiên, kinh tế, xã hội của tỉnh, đặc biệt là điều kiện đất đai và cơ cấu cây trồng.
        - Đảm bảo kịp thời, hiệu lực và hiệu quả trong triển khai thực hiện kế hoạch.
            + Điều 1: học tập tốt, lao động tốt
            + Điều 2: đoàn kết tốt, kỷ luật tốt
            + Điều 3: giữ gìn vệ sinh thật tốt'
       ========
        Output mẫu tiếng Anh giữ nguyên cấu trúc đề mục và các dấu '\n', '+', '-', kí tự xuống dòng và không sinh thêm kí tự '*', '**':
        'I. Purpose and Requirements:
        - Specify the sustainable development goals of the cassava industry according to Decision 1115/QD-BNN-TT of the Ministry of Agriculture and Rural Development.
        - Align with the natural, economic, and social conditions of the province, especially land conditions and crop structure.
        - Ensure timely, effective, and efficient implementation of the plan.
            + Point 1: Good learning, good work
            + Point 2: Good unity, good discipline
            + Point 3: Maintain very good hygiene.'
       =========
       Giờ đến lượt bạn sinh ra output:
        Input: '{input_}' 
    Output tiếng Anh giữ nguyên cấu trúc đề mục và các dấu '\n', '+', '-', kí tự xuống dòng và không sinh thêm kí tự '*', '**':
    """}],
    )
    return response.choices[0].message.content

In [7]:
os.mkdir('data_30k_35k')

In [7]:
import json
import os

for i in tqdm(range(len(dataset))):
    try:
        if f'{i}.json' in os.listdir('data_30k_35k'):
            continue
        ele = dataset[i]
        text = ele['text']
        if text is None:
            continue
        list_text = text.split('\n')
        doc_str = ''
        list_str = []
        for txt in list_text:
            doc_str += txt + '\n'
            if len(doc_str) > 600:
                list_str.append(doc_str)
                doc_str = ''
            # doc_str += txt + '\n'
        list_output = []
        for e in list_str:
            list_output.append({'vi' : e, 'en' : prompt(e)})
        with open(f'data_30k_35k/{i}.json', 'w') as outfile:
            json.dump(list_output,outfile)    
    except:
        continue

100%|██████████| 3887/3887 [02:30<00:00, 25.87it/s] 


In [ ]:
prompt(e)